# P000233DS — Qwen 2.5 7B Question LoRA Fine-tune
**EdGenAI Technologies | RMIT Capstone**

Fine-tunes Qwen/Qwen2.5-7B-Instruct on GraphRAG question generation data using QLoRA.

### Setup
1. Upload `graphrag_question_train.jsonl` and `graphrag_question_eval.jsonl` as a Kaggle Dataset named `graphrag-question-data`
2. Add the dataset to this notebook via **+ Add Data**
3. Set accelerator: **Settings → Accelerator → GPU T4 x2** (or P100)
4. Run All Cells

**Expected training time:** ~2–3 hours on T4 x1 for 2K samples × 3 epochs

In [ ]:
# Cell 1 — Install packages (run once, then RESTART SESSION if needed)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'       # single GPU — avoids DataParallel issues
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

%pip uninstall -y wandb -q
%pip install -q --force-reinstall 'protobuf==5.29.3'
%pip install -q \
    'transformers==4.46.3' \
    'peft==0.13.2' \
    'trl==0.12.1' \
    'accelerate>=0.34.0' \
    'bitsandbytes>=0.43.1' \
    'datasets>=2.19.0'

print('Packages installed. If this is your first run, restart the session now.')

In [ ]:
# Cell 2 — Imports + GPU check
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import json
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import transformers, peft

print(f'transformers : {transformers.__version__}')
print(f'peft         : {peft.__version__}')
print(f'torch        : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Enable GPU under Settings → Accelerator.')

In [ ]:
# Cell 3 — Config
# ── Paths ──────────────────────────────────────────────────────────────────
# After adding your dataset in Kaggle, it appears under /kaggle/input/<dataset-name>/
# Change DATASET_NAME if you named it differently when uploading
DATASET_NAME = 'graphrag-question-data'   # ← change if needed
TRAIN_FILE   = f'/kaggle/input/{DATASET_NAME}/graphrag_question_train.jsonl'
EVAL_FILE    = f'/kaggle/input/{DATASET_NAME}/graphrag_question_eval.jsonl'
OUTPUT_DIR   = '/kaggle/working/question_lora_checkpoints'
FINAL_DIR    = '/kaggle/working/question_lora'

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID     = 'Qwen/Qwen2.5-7B-Instruct'

# ── Training hyperparameters ───────────────────────────────────────────────
MAX_SEQ_LEN  = 1024     # safe for T4 16GB with 4-bit quant
BATCH_SIZE   = 1
GRAD_ACCUM   = 8        # effective batch = 8
EPOCHS       = 3
LR           = 1e-4
SEED         = 42

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
# Same targets as Shrinidhi's answer LoRA for consistency
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# Verify files exist
for f in [TRAIN_FILE, EVAL_FILE]:
    if not Path(f).exists():
        raise FileNotFoundError(
            f'{f} not found.\n'
            'Add your dataset via + Add Data → Your Datasets.\n'
            'Upload graphrag_question_train.jsonl and graphrag_question_eval.jsonl'
        )
print(f'Train file: {TRAIN_FILE} ✓')
print(f'Eval file:  {EVAL_FILE}  ✓')

In [ ]:
# Cell 4 — Load and inspect data
raw_train = load_dataset('json', data_files=TRAIN_FILE, split='train')
raw_eval  = load_dataset('json', data_files=EVAL_FILE,  split='train')

print(f'Train rows : {len(raw_train)}')
print(f'Eval rows  : {len(raw_eval)}')
print(f'Columns    : {raw_train.column_names}')

# Check distribution of question types and difficulties
from collections import Counter
qtypes = Counter(r['metadata']['question_type'] for r in raw_train)
diffs  = Counter(r['metadata']['difficulty']    for r in raw_train)
print(f'\nQuestion types : {dict(qtypes)}')
print(f'Difficulties   : {dict(diffs)}')

# Preview first sample
sample = raw_train[0]
print('\n── Sample messages ──')
for msg in sample['messages']:
    role = msg['role'].upper()
    content = msg['content'][:200] + '...' if len(msg['content']) > 200 else msg['content']
    print(f'[{role}] {content}\n')

In [ ]:
# Cell 5 — Tokenizer + format to text
torch.manual_seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_row(row):
    text = tokenizer.apply_chat_template(
        row['messages'],
        tokenize=False,
        add_generation_prompt=False,
    ) + tokenizer.eos_token
    return {'text': text}

train_formatted = raw_train.map(format_row, remove_columns=raw_train.column_names)
eval_formatted  = raw_eval.map(format_row,  remove_columns=raw_eval.column_names)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_tok = train_formatted.map(tokenize_fn, batched=True, remove_columns=['text'])
eval_tok  = eval_formatted.map(tokenize_fn,  batched=True, remove_columns=['text'])

lengths = [len(x) for x in train_tok['input_ids']]
print(f'Token lengths — min: {min(lengths)}  mean: {int(np.mean(lengths))}  max: {max(lengths)}')
truncated = sum(1 for l in lengths if l == MAX_SEQ_LEN)
print(f'Truncated samples (={MAX_SEQ_LEN} tokens): {truncated} / {len(lengths)}')
if truncated / len(lengths) > 0.1:
    print('[WARN] >10% of samples truncated. Consider increasing MAX_SEQ_LEN.')

In [ ]:
# Cell 6 — Load Qwen 2.5 7B in 4-bit + apply QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={'': 0},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Expected: ~1.2% trainable (all 7 modules), ~80-90M params

# Check VRAM after model load
vram_used = torch.cuda.memory_allocated(0) / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM used after model load: {vram_used:.1f} / {vram_total:.1f} GB')

In [ ]:
# Cell 7 — Data collator (fixed: strips non-tokenizer fields before padding)
def collator_with_labels(features):
    # Keep only input_ids + attention_mask — strips 'text' or metadata fields
    clean = [
        {'input_ids': f['input_ids'], 'attention_mask': f['attention_mask']}
        for f in features
    ]
    batch = tokenizer.pad(clean, padding=True, return_tensors='pt')
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask'] == 0] = -100   # mask padding from loss
    batch['labels'] = labels
    return batch

# Sanity check: collator works on 2 samples
sample_batch = collator_with_labels([train_tok[0], train_tok[1]])
print('Collator OK — batch keys:', list(sample_batch.keys()))
print('input_ids shape:', sample_batch['input_ids'].shape)
print('Non-masked labels:', (sample_batch['labels'] != -100).sum().item())

In [ ]:
# Cell 8 — Training
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    remove_unused_columns=False,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator_with_labels,
    processing_class=tokenizer,
)

print(f'Steps per epoch : {len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)}')
print(f'Total steps     : {len(train_tok) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}')
print(f'Effective batch : {BATCH_SIZE * GRAD_ACCUM}')
print('\nStarting training...')
torch.cuda.empty_cache()
trainer.train()

In [ ]:
# Cell 9 — Save adapter
Path(FINAL_DIR).mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Adapter saved to: {FINAL_DIR}')
print('Files:')
for f in sorted(Path(FINAL_DIR).iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:40s} {size_kb:8.1f} KB')

In [ ]:
# Cell 10 — Training loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_losses, label='Train loss', alpha=0.7, color='steelblue')
ax.plot(eval_steps,  eval_losses,  label='Eval loss',  marker='o', color='darkorange')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('QLoRA Fine-tuning — Qwen 2.5 7B Question LoRA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_loss.png', dpi=150)
plt.show()
print(f'Final train loss : {train_losses[-1]:.4f}')
print(f'Final eval loss  : {eval_losses[-1]:.4f}')
print(f'Best eval loss   : {min(eval_losses):.4f}')

In [ ]:
# Cell 11 — Qualitative evaluation: base vs fine-tuned on held-out samples
# Uses disable_adapter() to compare same model with/without LoRA weights
from peft import PeftModel
import random

random.seed(SEED)
N_SAMPLES   = 8    # number of examples to compare
MAX_NEW_TOK = 180

# Pick N random eval samples
eval_indices = random.sample(range(len(raw_eval)), min(N_SAMPLES, len(raw_eval)))
eval_samples = [raw_eval[i] for i in eval_indices]

def generate_question(mdl, msgs_without_assistant: list, max_new_tokens: int = MAX_NEW_TOK) -> str:
    input_ids = tokenizer.apply_chat_template(
        msgs_without_assistant,
        return_tensors='pt',
        add_generation_prompt=True,
    ).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

results = []
model.eval()
torch.cuda.empty_cache()

for i, sample in enumerate(eval_samples):
    # Input = system + user only (no assistant turn)
    prompt_msgs = sample['messages'][:2]
    reference   = sample['messages'][2]['content']  # ground truth
    qtype = sample['metadata']['question_type']
    diff  = sample['metadata']['difficulty']

    # Base model (LoRA disabled)
    with model.disable_adapter():
        base_q = generate_question(model, prompt_msgs)

    # Fine-tuned model (LoRA active)
    ft_q = generate_question(model, prompt_msgs)

    results.append({
        'idx': i+1, 'type': qtype, 'diff': diff,
        'reference': reference, 'base': base_q, 'finetuned': ft_q
    })
    print(f'[{i+1}/{N_SAMPLES}] {qtype} / {diff} ✓')

print('\nGeneration complete.')

In [ ]:
# Cell 12 — Print side-by-side comparison
SEP = '─' * 80
for r in results:
    print(f"\n{SEP}")
    print(f"Sample {r['idx']} | type: {r['type']} | difficulty: {r['diff']}")
    print(SEP)
    print(f"REFERENCE  : {r['reference']}")
    print(f"BASE MODEL : {r['base']}")
    print(f"FINE-TUNED : {r['finetuned']}")

In [ ]:
# Cell 13 — Quantitative: ROUGE-L score (no heavy dependencies)
# ROUGE-L measures longest common subsequence between predicted and reference
# Not a perfect metric for questions but gives a comparable number for your report

def lcs_length(a: str, b: str) -> int:
    a_tok, b_tok = a.lower().split(), b.lower().split()
    m, n = len(a_tok), len(b_tok)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a_tok[i-1] == b_tok[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_l(pred: str, ref: str) -> dict:
    lcs = lcs_length(pred, ref)
    ref_len  = len(ref.split())
    pred_len = len(pred.split())
    if ref_len == 0 or pred_len == 0:
        return {'p': 0, 'r': 0, 'f': 0}
    p = lcs / pred_len
    r = lcs / ref_len
    f = (2 * p * r / (p + r)) if (p + r) > 0 else 0
    return {'p': p, 'r': r, 'f': f}

base_rouges, ft_rouges = [], []
for r in results:
    base_rouges.append(rouge_l(r['base'],      r['reference'])['f'])
    ft_rouges.append(  rouge_l(r['finetuned'], r['reference'])['f'])

print('\n── ROUGE-L F1 (higher = more similar to reference question) ──')
print(f'Base model  : {np.mean(base_rouges):.4f} ± {np.std(base_rouges):.4f}')
print(f'Fine-tuned  : {np.mean(ft_rouges):.4f}  ± {np.std(ft_rouges):.4f}')
delta = np.mean(ft_rouges) - np.mean(base_rouges)
print(f'Delta       : {delta:+.4f} ({"improvement" if delta > 0 else "regression"})')
print('\nNote: ROUGE-L on open-ended questions has low ceiling.')
print('Use the qualitative comparison (Cell 12) as primary evidence.')

In [ ]:
# Cell 14 — Save all outputs as a zip for easy download
import zipfile

zip_path = '/kaggle/working/question_lora_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Adapter files
    for f in sorted(Path(FINAL_DIR).iterdir()):
        zf.write(f, f'question_lora/{f.name}')
    # Loss curve
    loss_png = Path('/kaggle/working/training_loss.png')
    if loss_png.exists():
        zf.write(loss_png, 'question_lora/training_loss.png')
    # Training log as JSON
    log_path = Path('/kaggle/working/training_log.json')
    log_path.write_text(json.dumps(trainer.state.log_history, indent=2))
    zf.write(log_path, 'question_lora/training_log.json')

zip_size = Path(zip_path).stat().st_size / 1e6
print(f'Output zip: {zip_path} ({zip_size:.1f} MB)')
print('Download from Kaggle → Output → question_lora_output.zip')
print('\nExtract to: finetune_data/question_lora/')
print('Then in config.yml set: use_question_lora: true')

## After downloading

1. Extract `question_lora_output.zip` → copy contents to `finetune_data/question_lora/`
2. In `config.yml`:
   ```yaml
   finetune:
     use_question_lora: true
   ```
3. Restart the API server: `python main_gr.py --mode full`
4. Re-run `dataset/generate_dataset.py` on the same learning objectives to get before/after judge scores for your capstone report.

### What to include in your capstone report
| Metric | Source |
|---|---|
| Trainable parameters | Cell 6 output |
| Final train / eval loss | Cell 10 output |
| Training loss curve | `training_loss.png` |
| ROUGE-L base vs fine-tuned | Cell 13 output |
| Qualitative comparison | Cell 12 output |
| Judge scores before/after | `dataset/generate_dataset.py` re-run |